In [ ]:
# ============================================================
# CELL 1 — ALL CALCULATIONS / FUNCTIONS / RESEARCH
# ============================================================
# Run this once. It loads data, builds the frozen Strategy 1,
# tests the post-1.5R reversal hypotheses, HTF conditioning,
# same-direction re-entry, and full-cycle economics.
# ============================================================

# ============================================================
# CELL 1 — CONFIG
# ============================================================

from QuantConnect.Research import QuantBook
from QuantConnect import Resolution, Market

import pandas as pd
import numpy as np
from datetime import datetime

qb = QuantBook()

def _market_attr(*names):
    for name in names:
        if hasattr(Market, name):
            return getattr(Market, name)
    raise AttributeError(names)

OANDA = _market_attr("Oanda", "OANDA")

DATA_START = datetime(2022, 1, 1)
DATA_END   = datetime(2026, 8, 15)

ASSETS = {
    "EURUSD": {
        "type": "forex",
        "market": OANDA,
        "maker_bps": 0.0,
        "taker_bps": 0.0,
        "fallback_spread_bps": 0.0,
    },
    "NAS100USD": {
        "type": "cfd",
        "market": OANDA,
        "maker_bps": 0.0,
        "taker_bps": 0.0,
        "fallback_spread_bps": 0.0,
    },
}

# Frozen live candidate
TIMEFRAME_MIN = 60
STRUCTURE_SIZE = 5
ATR_PERIOD = 14

FROZEN_MIN_DISPLACEMENT_ATR = 2.0
FROZEN_MIN_BROKEN_STRUCTURE_ATR = 1.0
FROZEN_MIN_CLOSE_THROUGH_ATR = 0.25
RETRACE = 0.618
PENDING_EXPIRY_BARS = 20
MAX_ENTRY_DISTANCE_ATR = 2.0
FROZEN_TP_R = 1.5

INTRABAR_PRIORITY = "stop_first"
CANCEL_ON_NEW_CHOCH = True
CANCEL_ON_OPPOSITE_BOS = True
INVALIDATE_ON_ORIGIN_BREAK = True
MAX_HOLD_BARS = 120

# HTF context is deliberately predefined, not optimised.
HTF_4H_STRUCTURE_SIZE = 5
HTF_DAILY_STRUCTURE_SIZE = 3

# After 1.5R is hit, inspect the next five trading days / 120 hours.
POST_TP_HORIZON_BARS = 120

CONTINUATION_R = [3.0, 5.0, 10.0]
REVERSAL_R = [1.0, 0.5, 0.0]

print("HTF research configured.")
print("Frozen markets:", list(ASSETS))
print("Frozen TP:", FROZEN_TP_R, "R")
print("Context: Daily + 4H")


# ============================================================
# CELL 2 — DATA LOAD + PANDAS RESAMPLING
# ============================================================

SECURITIES = {}

def add_asset(pair, spec):
    if spec["type"] == "forex":
        return qb.AddForex(pair, Resolution.Minute, spec["market"]).Symbol
    if spec["type"] == "cfd":
        return qb.AddCfd(pair, Resolution.Minute, spec["market"]).Symbol
    raise ValueError(spec["type"])

for pair, spec in ASSETS.items():
    SECURITIES[pair] = add_asset(pair, spec)
    print(pair, "subscribed")

def _flatten_history(history, pair):
    if history is None or history.empty:
        return pd.DataFrame()

    df = history.copy()

    if isinstance(df.index, pd.MultiIndex):
        try:
            df = df.loc[pair].copy()
        except Exception:
            df = df.reset_index()
            time_col = next(
                (c for c in ["time", "endtime", "index"] if c in df.columns),
                None
            )
            if time_col is None:
                raise
            df = df.set_index(time_col)

    df.columns = [str(c).lower().replace("_", "") for c in df.columns]

    idx = pd.to_datetime(df.index)
    if idx.tz is None:
        idx = idx.tz_localize("UTC")
    else:
        idx = idx.tz_convert("UTC")

    df.index = idx
    return df.sort_index()

def normalize_ohlc(history, pair):
    raw = _flatten_history(history, pair)

    if raw.empty:
        return raw

    out = pd.DataFrame(index=raw.index)

    bid = ["bidopen", "bidhigh", "bidlow", "bidclose"]
    ask = ["askopen", "askhigh", "asklow", "askclose"]

    has_quotes = all(c in raw.columns for c in bid + ask)

    if has_quotes:
        for c in bid + ask:
            out[c] = pd.to_numeric(raw[c], errors="coerce")

        out["open"]  = (out["bidopen"]  + out["askopen"])  / 2
        out["high"]  = (out["bidhigh"]  + out["askhigh"])  / 2
        out["low"]   = (out["bidlow"]   + out["asklow"])   / 2
        out["close"] = (out["bidclose"] + out["askclose"]) / 2

    else:
        for c in ["open", "high", "low", "close"]:
            out[c] = pd.to_numeric(raw[c], errors="coerce")

    return out.dropna(subset=["open", "high", "low", "close"])

def resample_ohlc(df, rule):
    agg = {
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
    }

    quote_cols = {
        "bidopen": "first",
        "bidhigh": "max",
        "bidlow": "min",
        "bidclose": "last",
        "askopen": "first",
        "askhigh": "max",
        "asklow": "min",
        "askclose": "last",
    }

    for c, how in quote_cols.items():
        if c in df.columns:
            agg[c] = how

    return (
        df.resample(rule)
        .agg(agg)
        .dropna(subset=["open", "high", "low", "close"])
    )

DATA_1M = {}
DATA = {}       # 1H, deliberately named DATA so inherited helpers work
DATA_4H = {}
DATA_1D = {}

for pair, symbol in SECURITIES.items():

    hist = qb.History(
        symbol,
        DATA_START,
        DATA_END,
        Resolution.Minute
    )

    df1 = normalize_ohlc(hist, pair)

    if df1.empty:
        print(pair, "NO DATA")
        continue

    DATA_1M[pair] = df1
    DATA[pair] = resample_ohlc(df1, "60min")
    DATA_4H[pair] = resample_ohlc(df1, "240min")
    DATA_1D[pair] = resample_ohlc(df1, "1D")

    print(
        pair,
        f"1m={len(df1):,}",
        f"1H={len(DATA[pair]):,}",
        f"4H={len(DATA_4H[pair]):,}",
        f"1D={len(DATA_1D[pair]):,}",
    )


# ============================================================
# CELL 3 — ATR + CAUSAL CHoCH/BOS DETECTION
# ============================================================

def add_atr(df, period=14):
    x = df.copy()

    prev = x["close"].shift(1)

    tr = pd.concat(
        [
            x["high"] - x["low"],
            (x["high"] - prev).abs(),
            (x["low"] - prev).abs(),
        ],
        axis=1,
    ).max(axis=1)

    x["atr"] = tr.rolling(
        period,
        min_periods=period
    ).mean()

    return x

def detect_structure_events(df, size):

    x = add_atr(df, ATR_PERIOD)

    if len(x) <= size + 2:
        return pd.DataFrame()

    highs = x["high"].to_numpy(float)
    lows = x["low"].to_numpy(float)
    closes = x["close"].to_numpy(float)
    times = x.index
    atrs = x["atr"].to_numpy(float)

    h = x["high"]
    l = x["low"]

    # Pivot at i-size only becomes knowable at current bar i.
    new_leg_high = (
        h.shift(size) > h.rolling(size).max()
    ).fillna(False).to_numpy(bool)

    new_leg_low = (
        l.shift(size) < l.rolling(size).min()
    ).fillna(False).to_numpy(bool)

    BEARISH_LEG = 0
    BULLISH_LEG = 1
    BEARISH = -1
    BULLISH = 1

    leg = BEARISH_LEG
    trend = 0

    pivot_high = None
    pivot_low = None

    pivot_high_crossed = True
    pivot_low_crossed = True

    events = []

    for i in range(size, len(x)):

        previous_leg = leg

        if new_leg_high[i]:
            leg = BEARISH_LEG
        elif new_leg_low[i]:
            leg = BULLISH_LEG

        if leg != previous_leg:
            src = i - size

            if leg == BULLISH_LEG:
                pivot_low = {
                    "price": float(lows[src]),
                    "pivot_time": times[src],
                    "available_time": times[i],
                }
                pivot_low_crossed = False

            else:
                pivot_high = {
                    "price": float(highs[src]),
                    "pivot_time": times[src],
                    "available_time": times[i],
                }
                pivot_high_crossed = False

        previous_close = closes[i - 1]
        close = closes[i]

        # Bullish break
        if pivot_high is not None and not pivot_high_crossed:

            level = pivot_high["price"]

            if previous_close <= level and close > level:

                tag = "CHoCH" if trend == BEARISH else "BOS"
                trend = BULLISH
                pivot_high_crossed = True

                if pivot_low is not None and pivot_low["available_time"] <= times[i]:

                    origin = float(pivot_low["price"])
                    extreme = float(highs[i])
                    atr = float(atrs[i]) if not np.isnan(atrs[i]) else np.nan

                    events.append({
                        "event_time": times[i],
                        "bar_pos": i,
                        "type": tag,
                        "direction": "bullish",
                        "broken_level": level,
                        "origin_price": origin,
                        "event_extreme": extreme,
                        "atr": atr,
                        "break_strength_atr": (
                            (close - level) / atr
                            if atr and atr > 0
                            else np.nan
                        ),
                    })

        # Bearish break
        if pivot_low is not None and not pivot_low_crossed:

            level = pivot_low["price"]

            if previous_close >= level and close < level:

                tag = "CHoCH" if trend == BULLISH else "BOS"
                trend = BEARISH
                pivot_low_crossed = True

                if pivot_high is not None and pivot_high["available_time"] <= times[i]:

                    origin = float(pivot_high["price"])
                    extreme = float(lows[i])
                    atr = float(atrs[i]) if not np.isnan(atrs[i]) else np.nan

                    events.append({
                        "event_time": times[i],
                        "bar_pos": i,
                        "type": tag,
                        "direction": "bearish",
                        "broken_level": level,
                        "origin_price": origin,
                        "event_extreme": extreme,
                        "atr": atr,
                        "break_strength_atr": (
                            (level - close) / atr
                            if atr and atr > 0
                            else np.nan
                        ),
                    })

    ev = pd.DataFrame(events)

    if not ev.empty:
        ev["initial_displacement_atr"] = np.where(
            ev["direction"] == "bullish",
            (ev["event_extreme"] - ev["origin_price"]) / ev["atr"],
            (ev["origin_price"] - ev["event_extreme"]) / ev["atr"],
        )

        ev["broken_structure_size"] = (
            ev["broken_level"] - ev["origin_price"]
        ).abs()

        ev["broken_structure_atr"] = (
            ev["broken_structure_size"]
            / ev["atr"].replace(0, np.nan)
        )

    return ev

EVENTS = {}

for pair, df in DATA.items():
    ev = detect_structure_events(df, STRUCTURE_SIZE)
    EVENTS[pair] = ev

    counts = ev["type"].value_counts().to_dict() if not ev.empty else {}

    print(
        pair,
        "events=", len(ev),
        "CHoCH=", counts.get("CHoCH",0),
        "BOS=", counts.get("BOS",0)
    )


# ============================================================
# CELL 4 — SUPERSESSION MAPS
# ============================================================

def build_supersession_maps(events):
    if events is None or events.empty:
        return {}

    e = events.sort_values("event_time").reset_index(drop=True)
    out = {}

    for i, row in e.iterrows():
        next_choch = None
        next_opposite_bos = None

        for j in range(i + 1, len(e)):
            r = e.iloc[j]

            if next_choch is None and r["type"] == "CHoCH":
                next_choch = r["event_time"]

            if (
                next_opposite_bos is None
                and r["type"] == "BOS"
                and r["direction"] != row["direction"]
            ):
                next_opposite_bos = r["event_time"]

            if next_choch is not None and next_opposite_bos is not None:
                break

        out[int(row["bar_pos"])] = {
            "next_choch": next_choch,
            "next_opposite_bos": next_opposite_bos,
        }

    return out

SUPERSESSION = {
    pair: build_supersession_maps(ev)
    for pair, ev in EVENTS.items()
}

print("Supersession maps built.")


# ============================================================
# CELL 5 — EXECUTION HELPERS
# ============================================================

def retracement_price(origin, extreme, direction, retrace):
    if direction == "bullish":
        return extreme - retrace * (extreme - origin)

    return extreme + retrace * (origin - extreme)

def pending_limit_filled(bar, direction, limit_price, fallback_spread_bps):
    half = fallback_spread_bps / 20000.0

    if direction == "bullish":

        if "asklow" in bar.index:
            return float(bar["asklow"]) <= limit_price

        return float(bar["low"]) * (1 + half) <= limit_price

    if "bidhigh" in bar.index:
        return float(bar["bidhigh"]) >= limit_price

    return float(bar["high"]) * (1 - half) >= limit_price

def executable_side(bar, direction):

    if direction == "bullish" and "bidlow" in bar.index:
        return {
            "high": float(bar["bidhigh"]),
            "low": float(bar["bidlow"]),
            "close": float(bar["bidclose"]),
        }

    if direction == "bearish" and "asklow" in bar.index:
        return {
            "high": float(bar["askhigh"]),
            "low": float(bar["asklow"]),
            "close": float(bar["askclose"]),
        }

    return {
        "high": float(bar["high"]),
        "low": float(bar["low"]),
        "close": float(bar["close"]),
    }

def fee_in_r(entry_price, exit_price, risk, entry_bps, exit_bps):
    if risk <= 0:
        return 0.0

    fee_price = (
        entry_price * entry_bps / 10000.0
        + exit_price * exit_bps / 10000.0
    )

    return fee_price / risk


# ============================================================
# CELL 6 — DYNAMIC CLOSE-ONLY FIB SIMULATOR
# ============================================================

def simulate_dynamic_trade(
    df,
    event,
    asset_spec,
    supersession_info,
    min_displacement_atr,
    retrace=RETRACE,
    pending_expiry_bars=PENDING_EXPIRY_BARS,
    max_entry_distance_atr=MAX_ENTRY_DISTANCE_ATR,
    slippage_bps=0.0,
):

    direction = event["direction"]
    event_pos = int(event["bar_pos"])
    event_time = event["event_time"]

    origin = float(event["origin_price"])
    atr = float(event["atr"])

    if np.isnan(atr) or atr <= 0:
        return None

    running_extreme = float(event["event_extreme"])

    armed = False
    armed_pos = None
    limit_price = None

    next_choch = supersession_info.get("next_choch")
    next_opposite_bos = supersession_info.get("next_opposite_bos")

    def displacement(extreme):
        if direction == "bullish":
            return (extreme - origin) / atr
        return (origin - extreme) / atr

    def fib(extreme):
        return retracement_price(
            origin,
            extreme,
            direction,
            retrace,
        )

    # Order can only become active after CHoCH candle closes.
    if displacement(running_extreme) >= min_displacement_atr:
        armed = True
        armed_pos = event_pos
        limit_price = fib(running_extreme)

    search_end = min(
        len(df),
        event_pos + 1 + pending_expiry_bars + 100
    )

    for pos in range(event_pos + 1, search_end):

        t = df.index[pos]
        bar = df.iloc[pos]

        # New structure supersedes pending setup.
        if (
            CANCEL_ON_NEW_CHOCH
            and next_choch is not None
            and t >= next_choch
        ):
            return {
                "status":"cancel_new_choch",
                "event_time":event_time,
                "armed":armed,
                "filled":False,
            }

        if (
            CANCEL_ON_OPPOSITE_BOS
            and next_opposite_bos is not None
            and t >= next_opposite_bos
        ):
            return {
                "status":"cancel_opposite_bos",
                "event_time":event_time,
                "armed":armed,
                "filled":False,
            }

        # Existing limit is live during this bar.
        if armed and limit_price is not None:

            if pos - armed_pos > pending_expiry_bars:
                return {
                    "status":"cancel_expired",
                    "event_time":event_time,
                    "armed":True,
                    "filled":False,
                }

            if pending_limit_filled(
                bar,
                direction,
                limit_price,
                asset_spec["fallback_spread_bps"],
            ):

                frozen_extreme = running_extreme

                slip = slippage_bps / 10000.0

                entry_price = (
                    limit_price
                    * (
                        1 + slip
                        if direction == "bullish"
                        else 1 - slip
                    )
                )

                stop = origin
                target = frozen_extreme

                risk = abs(entry_price - stop)

                if risk <= 0:
                    return None

                future = df.iloc[
                    pos:
                    min(len(df), pos + MAX_HOLD_BARS)
                ]

                outcome = None
                exit_price = None
                exit_time = None
                exit_fee_bps = asset_spec["taker_bps"]

                for exit_t, exit_bar in future.iterrows():

                    side = executable_side(
                        exit_bar,
                        direction
                    )

                    if direction == "bullish":
                        sl_hit = side["low"] <= stop
                        tp_hit = side["high"] >= target
                    else:
                        sl_hit = side["high"] >= stop
                        tp_hit = side["low"] <= target

                    if sl_hit and tp_hit:
                        outcome = (
                            "sl"
                            if INTRABAR_PRIORITY == "stop_first"
                            else "tp"
                        )
                    elif sl_hit:
                        outcome = "sl"
                    elif tp_hit:
                        outcome = "tp"
                    else:
                        continue

                    exit_time = exit_t

                    if outcome == "sl":
                        exit_price = (
                            stop
                            * (
                                1 - slip
                                if direction == "bullish"
                                else 1 + slip
                            )
                        )
                        exit_fee_bps = asset_spec["taker_bps"]
                    else:
                        exit_price = (
                            target
                            * (
                                1 - slip
                                if direction == "bullish"
                                else 1 + slip
                            )
                        )
                        exit_fee_bps = asset_spec["maker_bps"]

                    break

                if outcome is None:

                    last = future.iloc[-1]
                    exit_time = future.index[-1]

                    if direction == "bullish" and "bidclose" in future.columns:
                        exit_price = float(last["bidclose"])
                    elif direction == "bearish" and "askclose" in future.columns:
                        exit_price = float(last["askclose"])
                    else:
                        exit_price = float(last["close"])

                    outcome = "timeout"
                    exit_fee_bps = asset_spec["taker_bps"]

                gross_r = (
                    (exit_price - entry_price) / risk
                    if direction == "bullish"
                    else (entry_price - exit_price) / risk
                )

                fees_r = fee_in_r(
                    entry_price,
                    exit_price,
                    risk,
                    asset_spec["maker_bps"],
                    exit_fee_bps,
                )

                net_r = gross_r - fees_r

                return {
                    "status":"filled",
                    "event_time":event_time,
                    "entry_time":t,
                    "exit_time":exit_time,
                    "armed":True,
                    "filled":True,
                    "entry":entry_price,
                    "stop":stop,
                    "target":target,
                    "risk_price":risk,
                    "frozen_extreme":frozen_extreme,
                    "gross_r":gross_r,
                    "fees_r":fees_r,
                    "net_r":net_r,
                    "winner":bool(net_r > 0),
                    "outcome":outcome,
                }

        # Origin broken before fill.
        if INVALIDATE_ON_ORIGIN_BREAK:

            if direction == "bullish" and float(bar["low"]) <= origin:
                return {
                    "status":"cancel_origin_break",
                    "event_time":event_time,
                    "armed":armed,
                    "filled":False,
                }

            if direction == "bearish" and float(bar["high"]) >= origin:
                return {
                    "status":"cancel_origin_break",
                    "event_time":event_time,
                    "armed":armed,
                    "filled":False,
                }

        # Candle CLOSE: update impulse extreme.
        new_extreme = False

        if direction == "bullish":

            e = float(bar["high"])

            if e > running_extreme:
                running_extreme = e
                new_extreme = True

        else:

            e = float(bar["low"])

            if e < running_extreme:
                running_extreme = e
                new_extreme = True

        current_disp = displacement(
            running_extreme
        )

        if (
            not armed
            and current_disp >= min_displacement_atr
        ):
            armed = True
            armed_pos = pos
            limit_price = fib(running_extreme)

        elif armed and new_extreme:
            # Cancel/replace only after candle closes.
            limit_price = fib(running_extreme)

        if armed and limit_price is not None:

            distance_atr = (
                abs(float(bar["close"]) - limit_price)
                / atr
            )

            if distance_atr > max_entry_distance_atr:
                return {
                    "status":"cancel_too_far",
                    "event_time":event_time,
                    "armed":True,
                    "filled":False,
                }

    return {
        "status":"unresolved",
        "event_time":event_time,
        "armed":armed,
        "filled":False,
    }

def summarize_dynamic(df):

    if df is None or df.empty:
        return {
            "events":0,
            "armed":0,
            "filled":0,
            "fill_rate":np.nan,
            "avg_r":np.nan,
            "median_r":np.nan,
            "win_rate":np.nan,
            "total_r":0.0,
        }

    events = len(df)
    armed = int(df["armed"].fillna(False).sum())

    filled = df[
        df["filled"].eq(True)
    ].copy()

    if filled.empty:
        return {
            "events":events,
            "armed":armed,
            "filled":0,
            "fill_rate":0.0,
            "avg_r":np.nan,
            "median_r":np.nan,
            "win_rate":np.nan,
            "total_r":0.0,
        }

    return {
        "events":events,
        "armed":armed,
        "filled":len(filled),
        "fill_rate":len(filled)/events,
        "avg_r":filled["net_r"].mean(),
        "median_r":filled["net_r"].median(),
        "win_rate":filled["winner"].eq(True).mean(),
        "total_r":filled["net_r"].sum(),
    }

print("Dynamic simulator loaded.")


# ============================================================
# CELL 7 — BUILD FROZEN ENTRY SETUPS
# ============================================================
# We first identify the exact same frozen signal entries.
# Exit policy is NOT involved in setup selection.
# ============================================================

def collect_frozen_entries():
    rows = []

    for pair, df in DATA.items():
        spec = ASSETS[pair]
        events = EVENTS[pair]

        choch = events[
            (events["type"] == "CHoCH")
            & (events["broken_structure_atr"] >= FROZEN_MIN_BROKEN_STRUCTURE_ATR)
            & (events["break_strength_atr"] >= FROZEN_MIN_CLOSE_THROUGH_ATR)
        ].copy()

        supers = SUPERSESSION[pair]

        for _, event in choch.iterrows():
            # Use the existing simulator only to establish the causal fill.
            # We ignore its eventual exit and rebuild exit logic later.
            res = simulate_dynamic_trade(
                df=df,
                event=event,
                asset_spec=spec,
                supersession_info=supers.get(int(event["bar_pos"]), {}),
                min_displacement_atr=FROZEN_MIN_DISPLACEMENT_ATR,
                retrace=RETRACE,
                pending_expiry_bars=PENDING_EXPIRY_BARS,
                max_entry_distance_atr=MAX_ENTRY_DISTANCE_ATR,
                slippage_bps=0.0,
            )

            if res is None or not res.get("filled", False):
                continue

            # Reconstruct entry risk from actual entry + structural origin.
            entry_time = pd.Timestamp(res["entry_time"])
            entry_pos = df.index.get_indexer([entry_time])[0]

            entry_price = float(res["entry"])
            stop_price = float(event["origin_price"])
            risk = abs(entry_price - stop_price)

            if risk <= 0:
                continue

            structural_target = float(res["target"])

            rows.append({
                "pair": pair,
                "event_time": pd.Timestamp(event["event_time"]),
                "entry_time": entry_time,
                "entry_pos": int(entry_pos),
                "direction": event["direction"],
                "entry_price": entry_price,
                "stop_price": stop_price,
                "risk_price": risk,
                "structural_target": structural_target,
            })

    return pd.DataFrame(rows).sort_values(
        ["entry_time", "pair"]
    ).reset_index(drop=True)

entries_df = collect_frozen_entries()

print("Frozen entries:", len(entries_df))


# ============================================================
# CELL 8 — BUILD 4H + DAILY STRUCTURE CONTEXT
# ============================================================

EVENTS_4H = {}
EVENTS_1D = {}

for pair in DATA:

    EVENTS_4H[pair] = detect_structure_events(
        DATA_4H[pair],
        HTF_4H_STRUCTURE_SIZE,
    )

    EVENTS_1D[pair] = detect_structure_events(
        DATA_1D[pair],
        HTF_DAILY_STRUCTURE_SIZE,
    )

    print(
        pair,
        "4H events:", len(EVENTS_4H[pair]),
        "| Daily events:", len(EVENTS_1D[pair])
    )

def htf_state_at(events, ts):
    if events is None or events.empty:
        return "unknown"

    known = events[
        events["event_time"] <= ts
    ]

    if known.empty:
        return "unknown"

    return (
        known
        .sort_values("event_time")
        .iloc[-1]["direction"]
    )

def classify_alignment(signal_direction, daily_state, h4_state):

    opposite = (
        "bearish"
        if signal_direction == "bullish"
        else "bullish"
    )

    if (
        daily_state == signal_direction
        and h4_state == signal_direction
    ):
        return "aligned_both"

    if (
        daily_state == opposite
        and h4_state == opposite
    ):
        return "opposed_both"

    if daily_state == "unknown" or h4_state == "unknown":
        return "unknown"

    return "mixed"

print("HTF context states ready.")


# ============================================================
# CELL 9 — FIXED 1.5R + POST-TP PATH ANALYSER
# ============================================================

def price_at_r(entry, risk, direction, r_value):
    if direction == "bullish":
        return entry + r_value * risk
    return entry - r_value * risk

def analyse_trade_path(df, row):

    entry_pos = int(row["entry_pos"])
    entry = float(row["entry_price"])
    stop = float(row["stop_price"])
    risk = float(row["risk_price"])
    direction = row["direction"] if "direction" in row.index else row["signal_direction"]

    tp = price_at_r(
        entry,
        risk,
        direction,
        FROZEN_TP_R,
    )

    # Existing 1.5R trade horizon.
    trade_future = df.iloc[
        entry_pos:
        min(len(df), entry_pos + MAX_HOLD_BARS)
    ]

    tp_time = None
    trade_r = None
    trade_exit_time = None

    for t, bar in trade_future.iterrows():

        side = executable_side(bar, direction)

        if direction == "bullish":
            sl_hit = side["low"] <= stop
            tp_hit = side["high"] >= tp
        else:
            sl_hit = side["high"] >= stop
            tp_hit = side["low"] <= tp

        if sl_hit and tp_hit:
            event = "sl" if INTRABAR_PRIORITY == "stop_first" else "tp"
        elif sl_hit:
            event = "sl"
        elif tp_hit:
            event = "tp"
        else:
            event = None

        if event == "sl":
            trade_r = -1.0
            trade_exit_time = t
            break

        if event == "tp":
            trade_r = FROZEN_TP_R
            trade_exit_time = t
            tp_time = t
            break

    if trade_r is None:
        last = trade_future.iloc[-1]

        if direction == "bullish" and "bidclose" in trade_future.columns:
            close = float(last["bidclose"])
        elif direction == "bearish" and "askclose" in trade_future.columns:
            close = float(last["askclose"])
        else:
            close = float(last["close"])

        trade_r = (
            (close - entry) / risk
            if direction == "bullish"
            else (entry - close) / risk
        )

        trade_exit_time = trade_future.index[-1]

    result = {
        "trade_r": float(trade_r),
        "winner": bool(trade_r > 0),
        "tp_hit": bool(tp_time is not None),
        "tp_time": tp_time,
        "trade_exit_time": trade_exit_time,
        "post_tp_mfe_r": np.nan,
    }

    for r in CONTINUATION_R:
        ri = int(r)
        result[f"reached_{ri}r"] = False
        result[f"reached_{ri}r_before_be"] = False

    result["reversed_to_1r"] = False
    result["reversed_to_0p5r"] = False
    result["reversed_to_entry"] = False

    if tp_time is None:
        return result

    start_pos = df.index.get_indexer([tp_time])[0]

    post = df.iloc[
        start_pos:
        min(
            len(df),
            start_pos + POST_TP_HORIZON_BARS + 1
        )
    ]

    max_favourable_r = FROZEN_TP_R

    be_has_been_hit = False

    for t, bar in post.iterrows():

        side = executable_side(bar, direction)

        if direction == "bullish":
            favourable_r = (side["high"] - entry) / risk
            adverse_r = (side["low"] - entry) / risk
        else:
            favourable_r = (entry - side["low"]) / risk
            adverse_r = (entry - side["high"]) / risk

        max_favourable_r = max(
            max_favourable_r,
            favourable_r,
        )

        # Conservative ordering for conditional runner:
        # if target and BE occur in same bar, BE wins.
        bar_hits_be = adverse_r <= 0

        for r in CONTINUATION_R:
            ri = int(r)

            if favourable_r >= r:
                result[f"reached_{ri}r"] = True

                if (
                    not be_has_been_hit
                    and not bar_hits_be
                ):
                    result[
                        f"reached_{ri}r_before_be"
                    ] = True

        if adverse_r <= 1.0:
            result["reversed_to_1r"] = True

        if adverse_r <= 0.5:
            result["reversed_to_0p5r"] = True

        if adverse_r <= 0.0:
            result["reversed_to_entry"] = True
            be_has_been_hit = True

    result["post_tp_mfe_r"] = max_favourable_r

    return result

print("Post-TP path analyser loaded.")


# ============================================================
# CELL 10 — BUILD HTF-CONTEXT LEDGER
# ============================================================

context_rows = []

for _, row in entries_df.iterrows():

    pair = row["pair"]
    entry_time = pd.Timestamp(row["entry_time"])

    daily_state = htf_state_at(
        EVENTS_1D[pair],
        entry_time,
    )

    h4_state = htf_state_at(
        EVENTS_4H[pair],
        entry_time,
    )

    alignment = classify_alignment(
        row["direction"],
        daily_state,
        h4_state,
    )

    path = analyse_trade_path(
        DATA[pair],
        row,
    )

    context_rows.append({
        "pair": pair,
        "entry_time": entry_time,
        "year": entry_time.year,
        "period": (
            "selection_2022_25"
            if entry_time.year <= 2025
            else "holdout_2026"
        ),
        "signal_direction": row["direction"],
        "daily_state": daily_state,
        "h4_state": h4_state,
        "alignment": alignment,
        **path,
    })

context_df = (
    pd.DataFrame(context_rows)
    .sort_values(["entry_time", "pair"])
    .reset_index(drop=True)
)

print("Context trades:", len(context_df))


# ============================================================
# CELL 11 — EXHAUSTION REVERSAL SIMULATOR
# ============================================================
# Reverse immediately when Strategy 1 hits +1.5R.
#
# Reversal targets are expressed in ORIGINAL-TRADE R coordinates:
#   target +1.0R  = 0.5R retracement from the 1.5R TP
#   target +0.5R  = 1.0R retracement from the TP
#   target  0.0R  = full retracement back to original entry
#
# Reversal stops are continuation levels in ORIGINAL-TRADE R:
#   +2.0R / +2.5R / +3.0R
#
# This is not yet a live rule. It maps the expectancy surface.
# ============================================================

REVERSAL_TARGET_ORIGINAL_R = [1.0, 0.5, 0.0]
REVERSAL_STOP_ORIGINAL_R = [2.0, 2.5, 3.0]

def original_r_price(entry, risk, direction, r_value):
    if direction == "bullish":
        return entry + r_value * risk
    return entry - r_value * risk

def simulate_post_tp_reversal(
    df,
    row,
    target_original_r,
    stop_original_r,
    horizon_bars=120,
):
    if not row["tp_hit"] or pd.isna(row["tp_time"]):
        return None

    entry = float(row["entry_price"])
    risk = float(row["risk_price"])
    direction = row["direction"] if "direction" in row.index else row["signal_direction"]

    tp_time = pd.Timestamp(row["tp_time"])
    start_pos = df.index.get_indexer([tp_time])[0]

    reversal_entry = original_r_price(
        entry, risk, direction, FROZEN_TP_R
    )

    reversal_target = original_r_price(
        entry, risk, direction, target_original_r
    )

    reversal_stop = original_r_price(
        entry, risk, direction, stop_original_r
    )

    reversal_risk = abs(reversal_stop - reversal_entry)
    reversal_reward = abs(reversal_entry - reversal_target)

    if reversal_risk <= 0:
        return None

    rr = reversal_reward / reversal_risk

    future = df.iloc[
        start_pos:
        min(len(df), start_pos + horizon_bars + 1)
    ]

    for t, bar in future.iterrows():

        side = executable_side(bar, direction)

        # Original bullish trade => reversal is short.
        if direction == "bullish":
            stop_hit = side["high"] >= reversal_stop
            target_hit = side["low"] <= reversal_target

        # Original bearish trade => reversal is long.
        else:
            stop_hit = side["low"] <= reversal_stop
            target_hit = side["high"] >= reversal_target

        # Conservative if both happen in same bar.
        if stop_hit:
            return {
                "reversal_r": -1.0,
                "reversal_win": False,
                "reversal_exit_time": t,
                "reversal_rr": rr,
                "reversal_outcome": "stop",
            }

        if target_hit:
            return {
                "reversal_r": rr,
                "reversal_win": True,
                "reversal_exit_time": t,
                "reversal_rr": rr,
                "reversal_outcome": "target",
            }

    return {
        "reversal_r": np.nan,
        "reversal_win": False,
        "reversal_exit_time": future.index[-1],
        "reversal_rr": rr,
        "reversal_outcome": "timeout",
    }

print("Reversal simulator loaded.")


# ============================================================
# CELL 12 — REVERSAL EXPECTANCY SURFACE
# ============================================================

rev_rows = []

tp_only = context_df[
    context_df["tp_hit"].eq(True)
].copy()

# Attach original frozen entry information needed by reversal simulator.
entry_lookup = entries_df.set_index(
    ["pair", "entry_time"]
)

for _, row in tp_only.iterrows():

    key = (row["pair"], row["entry_time"])
    erow = entry_lookup.loc[key]

    enriched = row.copy()
    enriched["entry_price"] = erow["entry_price"]
    enriched["risk_price"] = erow["risk_price"]
    # Normalize naming for downstream helpers.
    if "direction" not in enriched.index:
        enriched["direction"] = enriched["signal_direction"]

    df = DATA[row["pair"]]

    for target_r in REVERSAL_TARGET_ORIGINAL_R:
        for stop_r in REVERSAL_STOP_ORIGINAL_R:

            res = simulate_post_tp_reversal(
                df,
                enriched,
                target_original_r=target_r,
                stop_original_r=stop_r,
            )

            if res is None:
                continue

            rev_rows.append({
                "pair": row["pair"],
                "entry_time": row["entry_time"],
                "period": row["period"],
                "alignment": row["alignment"],
                "signal_direction": row["signal_direction"],
                "daily_state": row["daily_state"],
                "h4_state": row["h4_state"],
                "target_original_r": target_r,
                "stop_original_r": stop_r,
                **res,
            })

reversal_trades_df = pd.DataFrame(rev_rows)

reversal_surface_df = (
    reversal_trades_df
    .dropna(subset=["reversal_r"])
    .groupby(
        [
            "period",
            "target_original_r",
            "stop_original_r",
        ],
        as_index=False,
    )
    .agg(
        trades=("reversal_r", "size"),
        avg_r=("reversal_r", "mean"),
        win_rate=("reversal_win", "mean"),
        total_r=("reversal_r", "sum"),
        avg_rr=("reversal_rr", "mean"),
    )
)



# ============================================================
# CELL 13 — REVERSAL BY HTF CONTEXT
# ============================================================

reversal_alignment_df = (
    reversal_trades_df
    .dropna(subset=["reversal_r"])
    .groupby(
        [
            "period",
            "target_original_r",
            "stop_original_r",
            "alignment",
        ],
        as_index=False,
    )
    .agg(
        trades=("reversal_r", "size"),
        avg_r=("reversal_r", "mean"),
        win_rate=("reversal_win", "mean"),
        total_r=("reversal_r", "sum"),
    )
)

reversal_market_df = (
    reversal_trades_df
    .dropna(subset=["reversal_r"])
    .groupby(
        [
            "period",
            "target_original_r",
            "stop_original_r",
            "pair",
        ],
        as_index=False,
    )
    .agg(
        trades=("reversal_r", "size"),
        avg_r=("reversal_r", "mean"),
        win_rate=("reversal_win", "mean"),
        total_r=("reversal_r", "sum"),
    )
)



# ============================================================
# CELL 14 — FREEZE BEST REVERSAL USING 2022-2025 ONLY
# ============================================================

selection_rev = reversal_surface_df[
    reversal_surface_df["period"].eq("selection_2022_25")
].copy()

# Require useful sample size.
selection_rev = selection_rev[
    selection_rev["trades"] >= 50
].copy()

selection_rev["positive"] = (
    selection_rev["avg_r"] > 0
)

selection_rev = selection_rev.sort_values(
    ["positive", "avg_r", "trades"],
    ascending=[False, False, False],
)

if selection_rev.empty:
    frozen_reversal = None
    print("No reversal configuration met the sample threshold.")
else:
    frozen_reversal = selection_rev.iloc[0]

    print(
        "FROZEN REVERSAL FROM 2022-25:",
        f"target original {frozen_reversal['target_original_r']}R",
        f"| stop original {frozen_reversal['stop_original_r']}R",
        f"| avgR {frozen_reversal['avg_r']:.4f}",
        f"| n {int(frozen_reversal['trades'])}",
    )

    holdout_rev = reversal_surface_df[
        (reversal_surface_df["period"] == "holdout_2026")
        & (
            reversal_surface_df["target_original_r"]
            == frozen_reversal["target_original_r"]
        )
        & (
            reversal_surface_df["stop_original_r"]
            == frozen_reversal["stop_original_r"]
        )
    ]

    print("\n2026 HOLDOUT:")


# ============================================================
# CELL 15 — RE-ENTRY SETUP
# ============================================================
# Re-entry is only investigated after the reversal leg has completed.
#
# We deliberately use a simple, causal rule:
# after the reversal trade reaches its target, wait for the NEXT
# 1H CHoCH in the ORIGINAL Strategy-1 direction.
#
# Then apply the exact frozen Strategy-1 entry machinery again:
# dynamic 61.8% retracement, same quality filters, structural stop,
# 1.5R TP.
#
# This avoids inventing a special re-entry rule.
# ============================================================

def next_same_direction_choch(
    pair,
    after_time,
    original_direction,
):
    ev = EVENTS[pair]

    if ev is None or ev.empty:
        return None

    candidates = ev[
        (ev["event_time"] > after_time)
        & (ev["type"] == "CHoCH")
        & (ev["direction"] == original_direction)
        & (
            ev["broken_structure_atr"]
            >= FROZEN_MIN_BROKEN_STRUCTURE_ATR
        )
        & (
            ev["break_strength_atr"]
            >= FROZEN_MIN_CLOSE_THROUGH_ATR
        )
    ]

    if candidates.empty:
        return None

    return candidates.sort_values(
        "event_time"
    ).iloc[0]

print("Re-entry signal finder loaded.")


# ============================================================
# CELL 16 — SIMULATE SAME-DIRECTION RE-ENTRY
# ============================================================

reentry_rows = []

if frozen_reversal is not None:

    tgt = frozen_reversal["target_original_r"]
    stp = frozen_reversal["stop_original_r"]

    chosen_reversals = reversal_trades_df[
        (reversal_trades_df["target_original_r"] == tgt)
        & (reversal_trades_df["stop_original_r"] == stp)
        & (reversal_trades_df["reversal_outcome"] == "target")
    ].copy()

    supers_by_pair = {
        pair: build_supersession_map(EVENTS[pair])
        for pair in EVENTS
    }

    for _, rr in chosen_reversals.iterrows():

        pair = rr["pair"]

        re_event = next_same_direction_choch(
            pair,
            rr["reversal_exit_time"],
            rr["signal_direction"],
        )

        if re_event is None:
            continue

        df = DATA[pair]
        spec = ASSETS[pair]

        res = simulate_dynamic_trade(
            df=df,
            event=re_event,
            asset_spec=spec,
            supersession_info=supers_by_pair[pair].get(
                int(re_event["bar_pos"]),
                {}
            ),
            min_displacement_atr=FROZEN_MIN_DISPLACEMENT_ATR,
            retrace=RETRACE,
            pending_expiry_bars=PENDING_EXPIRY_BARS,
            max_entry_distance_atr=MAX_ENTRY_DISTANCE_ATR,
            slippage_bps=0.0,
        )

        if res is None or not res.get("filled", False):
            continue

        # Convert frozen structural exit to fixed 1.5R exit.
        entry_price = float(res["entry"])
        stop_price = float(res["stop"])
        risk = float(res["risk_price"])

        target_15 = original_r_price(
            entry_price,
            risk,
            re_event["direction"],
            FROZEN_TP_R,
        )

        entry_time = pd.Timestamp(res["entry_time"])
        entry_pos = df.index.get_indexer([entry_time])[0]

        temp_row = pd.Series({
            "entry_pos": entry_pos,
            "entry_price": entry_price,
            "stop_price": stop_price,
            "risk_price": risk,
            "direction": re_event["direction"],
        })

        # Minimal fixed 1.5R simulator.
        future = df.iloc[
            entry_pos:
            min(len(df), entry_pos + MAX_HOLD_BARS)
        ]

        reentry_r = None
        reentry_exit = None

        for t, bar in future.iterrows():

            side = executable_side(
                bar,
                re_event["direction"],
            )

            if re_event["direction"] == "bullish":
                sl_hit = side["low"] <= stop_price
                tp_hit = side["high"] >= target_15
            else:
                sl_hit = side["high"] >= stop_price
                tp_hit = side["low"] <= target_15

            if sl_hit:
                reentry_r = -1.0
                reentry_exit = t
                break

            if tp_hit:
                reentry_r = FROZEN_TP_R
                reentry_exit = t
                break

        if reentry_r is None:
            last = future.iloc[-1]

            if (
                re_event["direction"] == "bullish"
                and "bidclose" in future.columns
            ):
                close = float(last["bidclose"])
            elif (
                re_event["direction"] == "bearish"
                and "askclose" in future.columns
            ):
                close = float(last["askclose"])
            else:
                close = float(last["close"])

            reentry_r = (
                (close - entry_price) / risk
                if re_event["direction"] == "bullish"
                else (entry_price - close) / risk
            )

            reentry_exit = future.index[-1]

        reentry_rows.append({
            "pair": pair,
            "original_entry_time": rr["entry_time"],
            "reversal_exit_time": rr["reversal_exit_time"],
            "reentry_signal_time": re_event["event_time"],
            "reentry_entry_time": entry_time,
            "period": rr["period"],
            "alignment": rr["alignment"],
            "reentry_r": reentry_r,
            "reentry_win": reentry_r > 0,
            "reentry_exit_time": reentry_exit,
        })

reentry_df = pd.DataFrame(reentry_rows)

if reentry_df.empty:
    reentry_summary_df = pd.DataFrame()
else:
    reentry_summary_df = (
        reentry_df
        .groupby(
            ["period", "alignment"],
            as_index=False,
        )
        .agg(
            trades=("reentry_r", "size"),
            avg_r=("reentry_r", "mean"),
            win_rate=("reentry_win", "mean"),
            total_r=("reentry_r", "sum"),
        )
    )



# ============================================================
# CELL 17 — FULL CYCLE ECONOMICS
# ============================================================
# Strategy 1 winner = +1.5R
# Reversal trade = simulated frozen reversal
# Re-entry = only where a new same-direction CHoCH later fills
#
# This is diagnostic because the legs do not necessarily all occur
# on every original winner.
# ============================================================

cycle_rows = []

if frozen_reversal is not None:

    tgt = frozen_reversal["target_original_r"]
    stp = frozen_reversal["stop_original_r"]

    chosen = reversal_trades_df[
        (reversal_trades_df["target_original_r"] == tgt)
        & (reversal_trades_df["stop_original_r"] == stp)
    ].copy()

    reentry_lookup = {}
    if not reentry_df.empty:
        for _, r in reentry_df.iterrows():
            reentry_lookup[
                (r["pair"], r["original_entry_time"])
            ] = r

    for _, r in chosen.iterrows():

        if pd.isna(r["reversal_r"]):
            continue

        key = (r["pair"], r["entry_time"])
        re = reentry_lookup.get(key)

        cycle_r = FROZEN_TP_R + r["reversal_r"]
        has_reentry = re is not None

        if has_reentry:
            cycle_r += re["reentry_r"]

        cycle_rows.append({
            "pair": r["pair"],
            "entry_time": r["entry_time"],
            "period": r["period"],
            "alignment": r["alignment"],
            "strategy1_r": FROZEN_TP_R,
            "reversal_r": r["reversal_r"],
            "has_reentry": has_reentry,
            "reentry_r": (
                re["reentry_r"]
                if has_reentry
                else np.nan
            ),
            "cycle_r": cycle_r,
        })

cycle_df = pd.DataFrame(cycle_rows)

if cycle_df.empty:
    cycle_summary_df = pd.DataFrame()
else:
    cycle_summary_df = (
        cycle_df
        .groupby(
            ["period"],
            as_index=False,
        )
        .agg(
            cycles=("cycle_r", "size"),
            avg_cycle_r=("cycle_r", "mean"),
            total_cycle_r=("cycle_r", "sum"),
            pct_with_reentry=("has_reentry", "mean"),
            avg_reversal_r=("reversal_r", "mean"),
            avg_reentry_r=("reentry_r", "mean"),
        )
    )


In [ ]:
# ============================================================
# CELL 2 — RESULTS ONLY
# ============================================================
# Run after Cell 1 and paste this output back into ChatGPT.
# ============================================================

# ============================================================
# CELL 18 — FINAL COPY / PASTE OUTPUT
# ============================================================

def compact(title, df, rows=50):
    print("\n" + "=" * 110)
    print(title)
    print("=" * 110)

    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        print("EMPTY")
        return

    with pd.option_context(
        "display.max_rows", rows,
        "display.max_columns", None,
        "display.width", 360,
        "display.float_format", lambda v: f"{v:.4f}",
    ):
        print(df.head(rows).to_string(index=False))

print("#" * 110)
print("# CHoCH V10 — EXHAUSTION / REVERSAL / RE-ENTRY REVIEW")
print("#" * 110)

print(
    "Frozen Strategy 1:",
    "EURUSD + NAS100 | 60m CHoCH | 61.8% entry | "
    "structural SL | 1.5R TP | no runner"
)

compact(
    "1. REVERSAL EXPECTANCY SURFACE",
    reversal_surface_df.sort_values(
        ["period", "avg_r"],
        ascending=[True, False],
    ),
)

if frozen_reversal is not None:
    print(
        "\nFrozen reversal from 2022-25:",
        f"target original={frozen_reversal['target_original_r']}R",
        f"stop original={frozen_reversal['stop_original_r']}R",
        f"avgR={frozen_reversal['avg_r']:.4f}",
        f"n={int(frozen_reversal['trades'])}",
    )

compact(
    "2. REVERSAL BY HTF ALIGNMENT",
    reversal_alignment_df,
)

compact(
    "3. REVERSAL BY MARKET",
    reversal_market_df,
)

compact(
    "4. SAME-DIRECTION RE-ENTRY RESULTS",
    reentry_summary_df,
)

compact(
    "5. FULL CYCLE ECONOMICS",
    cycle_summary_df,
)

print("\n" + "#" * 110)
print("# END V10 REVIEW")
print("#" * 110)
